# SC-03 — Fondements de Solidity avec C#

[Version Python](SC-03-Solidity-Basics-Python.ipynb) | [Fonctions et état](SC-04-Functions-State-CSharp.ipynb)

**Objectifs** : manipuler les types, variables et conversions Solidity ; compiler, déployer et interroger des contrats depuis C#. **Prérequis** : .NET Interactive, Nethereum, `solc` 0.8.28 et Anvil local. **Durée** : environ 2 h.

Lancer `anvil --host 127.0.0.1 --chain-id 31337` dans un terminal ; tous les comptes utilisés sont ceux du bac local. Les transactions sont minées avant les lectures.

In [1]:
#r "nuget:Nethereum.Web3, 7.1.0"
using System;

using System.Collections.Generic;

using System.Diagnostics;

using System.Linq;

using System.Numerics;

using System.Text.Json;

using Nethereum.Web3;

using Nethereum.Hex.HexTypes;

using Nethereum.Contracts;

// Vérifier le compilateur demandé avant de compiler toute source.
var versionProcess=Process.Start(new ProcessStartInfo("solc","--version"){RedirectStandardOutput=true});

var versionText=await versionProcess.StandardOutput.ReadToEndAsync();
 await versionProcess.WaitForExitAsync();

if(versionProcess.ExitCode!=0 || !versionText.Contains("0.8.28+commit.7893614a")) throw new Exception("solc 0.8.28 requis");

Console.WriteLine(versionText.Trim());

var url = "http://127.0.0.1:8545";

var web3 = new Web3(url);

var chain = await web3.Eth.ChainId.SendRequestAsync();

if (chain.Value != 31337) throw new Exception("Anvil local attendu (chaîne 31337)");

// Les comptes déverrouillés proviennent exclusivement du bac Anvil local.
var accounts = await web3.Eth.Accounts.SendRequestAsync();

var deployer = accounts[0];

Console.WriteLine($"Chaîne {chain.Value}, compte {deployer}");

async Task<Contract> Deploy(string source, string name, params object[] args) {
 // Standard JSON garde la source intégrale et demande explicitement ABI et bytecode.
 var input = JsonSerializer.Serialize(new { language="Solidity", sources=new Dictionary<string,object>{{"Example.sol",new {content=source}}}, settings=new {outputSelection=new Dictionary<string,object>{{"*",new Dictionary<string,object>{{"*",new[]{"abi","evm.bytecode.object"}}}}}} });

 var p=Process.Start(new ProcessStartInfo("solc","--standard-json"){RedirectStandardInput=true,RedirectStandardOutput=true,RedirectStandardError=true});

 await p.StandardInput.WriteAsync(input);
 p.StandardInput.Close();
 var output=await p.StandardOutput.ReadToEndAsync();
 var err=await p.StandardError.ReadToEndAsync();
 await p.WaitForExitAsync();

 if(p.ExitCode!=0) throw new Exception(err);

 using var doc=JsonDocument.Parse(output);

 if(doc.RootElement.TryGetProperty("errors",out var errors) && errors.EnumerateArray().Any(e=>e.GetProperty("severity").GetString()=="error")) throw new Exception(errors.ToString());

 // Sélectionner le contrat nommé : ni ABI ni bytecode ne doivent être absents.
 var c=doc.RootElement.GetProperty("contracts").GetProperty("Example.sol").GetProperty(name);
 var abi=c.GetProperty("abi").GetRawText();
 var bin=c.GetProperty("evm").GetProperty("bytecode").GetProperty("object").GetString();

 if(string.IsNullOrEmpty(bin) || abi=="[]") throw new Exception("ABI ou bytecode absent");

 // Le receipt de déploiement doit être miné et réussi avant toute lecture.
 var r=await web3.Eth.DeployContract.SendRequestAndWaitForReceiptAsync(abi,"0x"+bin,deployer,new HexBigInteger(5000000),null,args);

 if(r.Status.Value!=1 || string.IsNullOrEmpty(r.ContractAddress) || string.IsNullOrEmpty(r.TransactionHash) || r.BlockNumber==null) throw new Exception("Déploiement non miné");

 Console.WriteLine($"{name} déployé : {r.ContractAddress}, tx {r.TransactionHash}, gas {r.GasUsed.Value}");
 return web3.Eth.GetContract(abi,r.ContractAddress);

}
async Task Write(Contract c,string fn,params object[] args) {
 // Envoyer depuis le déployeur, attendre le receipt puis contrôler son statut.
 var r=await c.GetFunction(fn).SendTransactionAndWaitForReceiptAsync(deployer,new HexBigInteger(1000000),null,null,args);

 if(r.BlockNumber==null || string.IsNullOrEmpty(r.TransactionHash)) throw new Exception("Transaction non minée");

 if(r.Status.Value!=1) throw new InvalidOperationException($"Receipt status 0 : {r.TransactionHash}, bloc {r.BlockNumber.Value}, gas {r.GasUsed.Value}");

 Console.WriteLine($"{fn} : tx {r.TransactionHash}, bloc {r.BlockNumber.Value}, gas {r.GasUsed.Value}");

}
Console.WriteLine("Compilation standard-json et transactions configurées.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Nethereum.Web3, 7.1.0

solc, the solidity compiler commandline interface
Version: 0.8.28+commit.7893614a.Linux.g++


Chaîne 31337, compte 0xf39fd6e51aad88f6f4ce6ab8827279cfffb92266


Compilation standard-json et transactions configurées.


***

## 1. Structure d'un contrat

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract MonContrat {
    // Variables d'état
    // Fonctions
}
```

### Éléments

| Élément | Description |
|---------|-------------|
| `pragma` | Version du compilateur |
| `contract` | Mot-clé de definition |
| `{ }` | Corps du contrat |

Deux détails du squelette : le commentaire `// SPDX-License-Identifier: MIT` déclare la licence du code, et la ligne `pragma solidity` **épingue** le compilateur — le code ne compilera qu'avec une version compatible de la série déclarée. Le corps `{ }` accueille les variables d'état (persistées sur la blockchain) et les fonctions (la logique).

In [2]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Counter {
    uint256 public count;

    function increment() public {
        count += 1;
    }

    function getCount() public view returns (uint256) {
        return count;
    }
}
";

 var c = await Deploy(source,"Counter");


}

Counter déployé : 0x5fbdb2315678afecb367f032d93f642f64180aa3, tx 0xddfcd4cf3e0763e0efac47273368f9116d09f8b894d7b729191c243b27e5661e, gas 133935


La sortie précédente provient du contrat déployé sur Anvil. Les lectures consultent l’état après minage des transactions indiquées.

### De la compilation au contrat vivant

Le compilateur reçoit une source et produit une interface ABI et du bytecode. L’ABI décrit notamment les noms, paramètres et résultats des fonctions ; Nethereum s’en sert pour encoder les appels et décoder les réponses. Le bytecode est envoyé dans une transaction de création. Le receipt associe ensuite une adresse au contrat créé : cette adresse identifie une instance avec son propre état, tandis que la source décrit son comportement.

Une transaction acceptée par le nœud ne suffit pas à établir le résultat : le receipt doit porter un bloc et un statut réussi. Le programme contrôle ces champs avant d’utiliser l’adresse. Les prochains exemples compilent chacun le contrat nommé dans leur source ; cela évite de déployer par erreur un autre contrat si une source en contient plusieurs. Le gas affiché est celui relevé dans le receipt et varie avec le contrat et les règles du réseau.

***

## 2. Types de données (Value Types)

### 2.1 Booléens

En Solidity, les **value types** (booléens, entiers, adresses, bytes fixes) sont passés **par valeur** : chaque affectation copie la valeur, aucune référence partagée. Ils n'ont pas de « data location » à préciser — contrairement aux tableaux et structs (`storage`, `memory`, `calldata`), sujet du notebook SC-04.

In [3]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract BoolExample {
    bool public isActive = true;
    bool public isPaused = false;

    function toggle() public {
        isActive = !isActive;
    }

    function logicalAnd(bool a, bool b) public pure returns (bool) {
        return a && b;
    }

    function logicalOr(bool a, bool b) public pure returns (bool) {
        return a || b;
    }

    function logicalNot(bool a) public pure returns (bool) {
        return !a;
    }
}
";

 var c = await Deploy(source,"BoolExample");

Console.WriteLine($"isActive = {await c.GetFunction("isActive").CallAsync<bool>()}");
 await Write(c,"toggle");

Console.WriteLine($"après toggle = {await c.GetFunction("isActive").CallAsync<bool>()}; AND = {await c.GetFunction("logicalAnd").CallAsync<bool>(true,false)}; OR = {await c.GetFunction("logicalOr").CallAsync<bool>(true,false)}");

}

BoolExample déployé : 0xe7f1725e7734ce288f8367e1bb143e90bb3f0512, tx 0x4e9faeed2d3c8a995549d43fffcef3395c462309abdbf83a0e0d74f3b2493b22, gas 226095


isActive = True


toggle : tx 0x4001f9818329325981ad01444a37784377b99c84a0dfdc7e18e8e7b3f09ff3a8, bloc 3, gas 21631


après toggle = False; AND = False; OR = True


### Interpretation : Booléens et opérateurs logiques

**Résultat obtenu** : Les booléens supportent les opérations logiques classiques.

| Opérateur | Description | Exemple | Résultat |
|-----------|-------------|---------|----------|
| `&&` | ET logique | `true && false` | `false` |
| `\|\|` | OU logique | `true \|\| false` | `true` |
| `!` | NON logique | `!true` | `false` |

**Points clés** :
- `toggle()` inverse la valeur de `isActive` (true -> false, false -> true)
- Les opérateurs logiques suivent la logique booleenne classique
- Les booléens sont utilises dans les conditions (`require`, `if`, boucles)

### 2.2 Entiers

Les entiers sont les `uint` (non signés) et `int` (signés), déclinés en tailles de 8 à 256 bits par pas de 8 : `uint8` vaut de 0 à 255, `uint256` de 0 à 2^256−1, et `uint` sans précision est un alias de `uint256`. Toute opération qui déborde provoque un **revert** : la transaction est annulée, plus de dépassement silencieux — c'est le comportement par défaut du compilateur épinglé par la pragma de ce notebook.

In [4]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract IntExample {
    // Entiers non signes (positifs)
    uint8 public smallNumber;      // 0 a 255
    uint256 public bigNumber;      // 0 a 2^256-1
    uint public defaultUint;       // uint256 par defaut

    // Entiers signes (positifs et negatifs)
    int8 public signedSmall;       // -128 a 127
    int256 public signedBig;       // -2^255 a 2^255-1

    function add(uint a, uint b) public pure returns (uint) {
        return a + b;
    }

    function subtract(int a, int b) public pure returns (int) {
        return a - b;
    }

    function multiply(uint a, uint b) public pure returns (uint) {
        return a * b;
    }

    function modulo(uint a, uint b) public pure returns (uint) {
        return a % b;
    }
}
";

 var c = await Deploy(source,"IntExample");

foreach(var op in new[]{("add",100,200),("subtract",10,3),("multiply",7,6),("modulo",17,5)}) Console.WriteLine($"{op.Item1} = {await c.GetFunction(op.Item1).CallAsync<BigInteger>(new BigInteger(op.Item2),new BigInteger(op.Item3))}");

}

IntExample déployé : 0xcf7ed3acca5a467e9e704c703e8d87f634fb0fc9, tx 0xcb56c781abaf8225a75b051a3246afb3ef500da670aaaf8b5d288bcf947c96ff, gas 357952


add = 300


subtract = 7


multiply = 42


modulo = 2


### Interpretation : Types entiers

**Résultat obtenu** : Opérations arithmétiques de base sur les entiers Solidity.

| Opération | Résultat | Type |
|-----------|----------|------|
| `add(100, 200)` | 300 | Addition |
| `subtract(10, 3)` | 7 | Soustraction |
| `multiply(7, 6)` | 42 | Multiplication |
| `modulo(17, 5)` | 2 | Reste de division |

**Points clés** :
- `uint` est un alias pour `uint256` (entier non signé de 256 bits)
- `int256` peut représenter des nombres négatifs (-2^255 à 2^255-1)
- `uint8` stocke des valeurs de 0 à 255 seulement ; le gain de storage dépend du compactage des variables
- Solidity 0.8+ vérifie automatiquement les overflows/underflows (revert si erreur)

### 2.3 Addresses

Une `address` est un entier de 20 octets (160 bits) : l'identifiant d'un compte ou d'un contrat. Seul `address payable` peut **recevoir** des ETH (`transfer`, `send`, `call`) ; la conversion inverse est implicite, celle-ci doit être écrite explicitement (`payable(addr)`). Chaque adresse expose `.balance`, en wei (1 ETH = 10^18 wei).

In [5]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract AddressExample {
    address public owner;
    address payable public treasury;

    constructor() {
        owner = msg.sender;
    }

    function getBalance(address addr) public view returns (uint256) {
        return addr.balance;
    }

    function isContract(address addr) public view returns (bool) {
        uint256 size;
        assembly { size := extcodesize(addr) }
        return size > 0;
    }
}
";

 var c = await Deploy(source,"AddressExample");

var owner=await c.GetFunction("owner").CallAsync<string>();
 Console.WriteLine($"owner = {owner}; égal au déployeur : {string.Equals(owner,deployer,StringComparison.OrdinalIgnoreCase)}; solde Wei = {await c.GetFunction("getBalance").CallAsync<BigInteger>(deployer)}");

}

AddressExample déployé : 0xdc64a140aa3e981100a9beca4e685f962f0cf6c9, tx 0x3ec539e947f9c8f299c758a88091f1f5357e16a90e00ccd578b6423b9c2e4088, gas 237284


owner = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266; égal au déployeur : True; solde Wei = 9999998293718633131787


### Interpretation : Type address

**Résultat obtenu** : Le type `address` represente une adresse Ethereum (20 octets).

**Points clés** :
- L'adresse du deployeur est capturee dans le constructor via `msg.sender`
- `msg.sender` est une variable globale contenant l'adresse de l'appelant
- `addr.balance` retourne le solde en wei d'une adresse (1 ETH = 10^18 wei)
- Sur anvil (blockchain locale), tous les comptes demarrent avec 10000 ETH
- La distinction `address` vs `address payable` : seul `address payable` peut recevoir des ETH

### Adresses, comptes et contrats

Une adresse Ethereum occupe 20 octets. Elle peut désigner un compte contrôlé par une clé ou une instance de contrat. Lors de la création, `msg.sender` désigne l’émetteur de la transaction de déploiement ; le constructeur le copie dans `owner`. La lecture compare cette valeur au premier compte du bac local sans dépendre de la casse de son affichage hexadécimal.

`getBalance` renvoie une quantité en Wei, la plus petite unité de l’ETH. Il faut conserver cette quantité comme un entier : une conversion flottante peut arrondir de grandes valeurs. Le solde affiché dépend des transactions précédentes, notamment des frais de création des contrats. `isContract` consulte la taille du code associé à une adresse ; le code source montre comment cette inspection est réalisée dans l’EVM.

### 2.4 Bytes et String

`bytes32` est un tableau **fixe** de 32 octets — idéal pour hachés et identifiants de taille connue, il tient dans un seul slot de storage. `bytes` est le tableau **dynamique** d'octets. `string` est la chaîne UTF-8 du quotidien : sous le capot un `bytes` dynamique interprété comme de l'UTF-8. Manipuler une `string` coûte plus de gaz qu'un `bytes` de même taille (validation UTF-8) : stockez en `bytes` quand le contenu n'est pas du texte.

In [6]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract BytesExample {
    bytes32 public fixedData = ""hello"";
    bytes public dynamicData = ""world"";
    string public message = ""Hello Solidity"";

    function setFixed(bytes32 _data) public {
        fixedData = _data;
    }

    function setMessage(string memory _msg) public {
        message = _msg;
    }

    function getLength() public view returns (uint) {
        return bytes(message).length;
    }

    function concatenate(string memory a, string memory b) public pure returns (string memory) {
        return string(abi.encodePacked(a, b));
    }
}
";

 var c = await Deploy(source,"BytesExample");

Console.WriteLine($"message = {await c.GetFunction("message").CallAsync<string>()}; longueur = {await c.GetFunction("getLength").CallAsync<BigInteger>()}");
 await Write(c,"setMessage","Solidity rocks!");
 Console.WriteLine($"message = {await c.GetFunction("message").CallAsync<string>()}; concaténation = {await c.GetFunction("concatenate").CallAsync<string>("Hello"," World")}");

}

BytesExample déployé : 0x5fc8d32690cc91d4c39d9d3abcbd16989f875707, tx 0x3d782e8dc27959f83bbcc7b450ec649288c21b5f6f6948a1148dd9af7e0a6af8, gas 687748


message = Hello Solidity; longueur = 14


setMessage : tx 0xd5f9a70e3b3aa4334aa37f6a512535497593054eb827472666fb5729aa1934b9, bloc 7, gas 28220


message = Solidity rocks!; concaténation = Hello World


### Interpretation : Types bytes et string

**Résultat obtenu** : Manipulation de chaînes de caractères et tableaux d'octets.

| Type | Description | Exemple |
|------|-------------|---------|
| `bytes32` | Tableau fixe de 32 octets | Stockage efficace pour données de taille connue |
| `bytes` | Tableau dynamique d'octets | Taille variable, plus flexible |
| `string` | Chaîne UTF-8 | Encodee en bytes sous-jacent |

**Points clés** :
- `setMessage("Solidity rocks!")` modifie la variable d'état `message`
- `getLength()` convertit le string en bytes pour obtenir sa longueur
- `concatenate()` utilise `abi.encodePacked()` pour fusionner deux strings
- En Solidity, les strings sont des wrappers autour de bytes dynamiques

***

## 3. Variables

### 3.1 Variables d'état

Trois familles de variables coexistent : les variables d'**état** (corps du contrat, persistées en `storage`), les variables **locales** (dans une fonction, en mémoire, effacées à la fin de l'appel) et les variables **globales** (fournies par l'EVM : `msg.sender`, `block.timestamp`...). Le choix n'est pas stylistique : il change le coût en gaz et la durée de vie de la donnée.

In [7]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract StateVariables {
    // Stockees en permanence sur la blockchain
    uint256 public storedData = 42;
    address public owner;
    bool private initialized = false;

    // Visibilite:
    // public   - getter auto-genere, visible de partout
    // private  - uniquement ce contrat
    // internal - ce contrat + contrats enfants
    // external - uniquement depuis l'exterieur (fonctions seulement)

    constructor() {
        owner = msg.sender;
        initialized = true;
    }

    function setData(uint256 _data) public {
        storedData = _data;
    }

    function isInitialized() public view returns (bool) {
        return initialized;
    }
}
";

 var c = await Deploy(source,"StateVariables");

Console.WriteLine($"storedData = {await c.GetFunction("storedData").CallAsync<BigInteger>()}; owner = {await c.GetFunction("owner").CallAsync<string>()}; initialisé = {await c.GetFunction("isInitialized").CallAsync<bool>()}");
 await Write(c,"setData",new BigInteger(100));
 Console.WriteLine($"storedData = {await c.GetFunction("storedData").CallAsync<BigInteger>()}");

}

StateVariables déployé : 0xa513e6e4b8f2a923d98304ec87f64353c4d5c853, tx 0x5ef7dcfe677554c29974fe84f2e4e035f6505b771e1d50071a41c0794494538f, gas 229758


storedData = 42; owner = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266; initialisé = True


setData : tx 0x2160119063b71a1d4dd8f6d8102975181786b9407408d295800c42c35398a3c1, bloc 9, gas 26640


storedData = 100


### Interpretation : Variables d'état et persistance

**Résultat obtenu** : Les variables d'état sont stockees de facon permanente sur la blockchain.

| Aspect | Valeur initiale | Valeur finale | Observation |
|--------|----------------|---------------|-------------|
| `storedData` | 42 | 100 | Modifiée par `setData()` |
| `owner` | Deployer address | Identique | Définie dans le constructor |
| `initialized` | True | Identique | Change de false a true dans le constructor |

**Points clés** :
- Le constructor est exécuté UNE SEULE FOIS au déploiement du contrat
- La visibilité `public` génère automatiquement un getter (ex: `storedData()`)
- La visibilité `private` signifie que seul ce contrat peut accéder à la variable (pas de getter auto)
- Les variables d'état persistent entre les différents appels de fonctions

### 3.2 Variables locales

Une variable déclarée dans une fonction vit en **mémoire** : elle n'existe que le temps de l'appel, n'est jamais écrite sur la blockchain et ne coûte donc presque aucun gaz de stockage — c'est le lieu des calculs intermédiaires. La contrepartie : ce qui n'est pas remonté (return, ou écriture dans une variable d'état) est perdu à la fin de l'appel.

In [8]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract LocalVariables {
    uint256 public result;

    function calculate(uint a, uint b) public returns (uint256) {
        // Variables locales (en memoire, non stockees sur la blockchain)
        uint256 sum = a + b;
        uint256 product = a * b;

        // Seul le resultat final est stocke
        result = sum + product;
        return result;
    }

    function computeOnly(uint a, uint b) public pure returns (uint256 sum, uint256 product) {
        // pure : ni lecture ni ecriture sur la blockchain
        sum = a + b;
        product = a * b;
    }
}
";

 var c = await Deploy(source,"LocalVariables");

Console.WriteLine($"calculate(3,4) = {await c.GetFunction("calculate").CallAsync<BigInteger>(new BigInteger(3),new BigInteger(4))}");
 await Write(c,"calculate",new BigInteger(3),new BigInteger(4));
 Console.WriteLine($"result = {await c.GetFunction("result").CallAsync<BigInteger>()}");
 var values=await c.GetFunction("computeOnly").CallDecodingToDefaultAsync(new BigInteger(5),new BigInteger(6));
 Console.WriteLine($"sum = {values[0].Result}, product = {values[1].Result}");

}

LocalVariables déployé : 0x8a791620dd6260079bf849dc5567adc3f2fdc318, tx 0x3f7d5f190cc9cb98baf4ad85bfaba8e732b962160c51b24d8065f3e6f908ae6e, gas 205782


calculate(3,4) = 19


calculate : tx 0x84567988a353ae1529c7e471bf4e8e2c2f9ea528b4e537199d692229e3602d1d, bloc 11, gas 44983


result = 19


sum = 11, product = 30


### Interpretation : Variables locales vs état

**Résultat obtenu** : Les variables locales sont temporaires et stockees en mémoire, tandis que les variables d'état persistent sur la blockchain.

| Aspect | Variables locales | Variables d'état |
|--------|------------------|------------------|
| **Stockage** | Memoire (RAM) | Blockchain (storage) |
| **Durée** | Temporaire (efface après l'appel) | Permanente |
| **Coût gas** | Faible | Élevé |
| **Portee** | Uniquement dans la fonction | Accessible partout dans le contrat |

**Points clés** :
- `calculate(3, 4)` retourne 19 et stocke le résultat dans la variable d'état `result`
- `computeOnly(5, 6)` utilise le mot-clé `pure` : ni lecture ni ecriture du storage, donc moins couteux en gas
- Le mot-clé `pure` indique au compilateur que la fonction ne depend pas de l'état du contrat

### Lecture simulée et écriture persistante

L’appel de `calculate(3, 4)` par `eth_call` évalue la fonction et retourne son résultat sans enregistrer son affectation. La transaction suivante exécute la même fonction et persiste `result` : seule la lecture qui suit son receipt vérifie l’état durable. Cette distinction est utile pour prévoir un résultat, mais une lecture préalable ne garantit pas qu’une transaction ultérieure réussira dans un état qui a pu changer.

La fonction `computeOnly` renvoie deux valeurs ABI. Nethereum les décode dans l’ordre déclaré, sans conversion en nombre flottant : la somme et le produit restent des entiers de précision arbitraire côté C#. Les variables intermédiaires de calcul ne sont pas elles-mêmes sauvegardées. Seule l’affectation à `result` écrit dans le storage du contrat.

### 3.3 Variables globales

L'EVM injecte dans chaque appel des variables implicites, accessibles sans déclaration : `msg.sender` (adresse de l'appelant), `msg.value` (ETH envoyés), `block.timestamp`, `block.number`, `block.gaslimit`... Ce ne sont pas des arguments de fonction mais le **contexte d'exécution**. `msg.sender` fonde tout le contrôle d'accès (`require(msg.sender == owner)`) ; `block.timestamp` est manipulable par les mineurs dans une certaine marge — ne l'utilisez jamais comme source d'aléa.

In [9]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract GlobalVariables {
    event GlobalsRead(
        address sender,
        uint256 timestamp,
        uint256 blockNumber,
        uint256 gasLimit
    );

    function getGlobals() public returns (
        address sender,
        uint256 timestamp,
        uint256 blockNumber,
        uint256 gasLimit
    ) {
        sender = msg.sender;
        timestamp = block.timestamp;
        blockNumber = block.number;
        gasLimit = block.gaslimit;
        emit GlobalsRead(sender, timestamp, blockNumber, gasLimit);
    }

    function getMsgValue() public payable returns (uint256) {
        return msg.value;
    }
}
";

 var c = await Deploy(source,"GlobalVariables");

var values=await c.GetFunction("getGlobals").CallDecodingToDefaultAsync(deployer,null,null,Array.Empty<object>());
 Console.WriteLine(string.Join(", ",values.Select(x=>$"{x.Parameter.Name} = {x.Result}")));

}

GlobalVariables déployé : 0xb7f8bc63bbcad18155201308c8f3540b07f84f5e, tx 0x1c2a60a39e96067643e3ae11140c456f1eea04f39b50b4a0570cd1e5b7e0e30b, gas 148819


sender = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266, timestamp = 1795392981, blockNumber = 12, gasLimit = 30000000


### Interprétation : les variables globales de l’EVM

`msg.sender` identifie l’appelant : l’appel de lecture précise explicitement le compte du déployeur. Sans cette adresse, certains clients exécutent l’appel avec l’adresse zéro. `block.timestamp`, `block.number` et `block.gaslimit` sont fournis par le bloc courant ; ils changent d’un replay à l’autre. Ils ne constituent ni une horloge parfaitement fiable ni une source de hasard sûre. Le contrat déclare aussi un événement pour exposer ces informations dans les journaux lorsqu’une transaction exécute la fonction, tandis que l’appel de lecture ne persiste aucun événement.

`msg.value` indique le montant en Wei attaché à un appel payable. Pour envoyer de l’ETH, il faut une transaction et une fonction qui accepte cette valeur. Une lecture seule ne crée pas de transfert. Les types numériques EVM sont représentés ici par `BigInteger` afin de garder les valeurs entières exactes jusqu’à 256 bits.

***

## 4. Conversions de types

Solidity distingue conversion **implicite** (sûre, ex. `uint8` vers `uint256`) et conversion **explicite** (à manier avec prudence, ex. `int256` vers `uint256`). Règle pratique : toute conversion qui peut perdre de l'information (signe, taille) s'écrit explicitement et se réfléchit — c'est exactement ce que montre la section.

In [10]:
Console.WriteLine(@"
contract TypeConversions {
    // Conversion explicite
    function uintToInt(uint256 a) public pure returns (int256) {
        return int256(a);
    }

    // Conversion address to payable
    function toPayable(address addr) public pure returns (address payable) {
        return payable(addr);
    }

    // String to bytes
    function stringToBytes(string memory s) public pure returns (bytes memory) {
        return bytes(s);
    }

    // Address to uint (pour comparaison)
    function addressToUint(address addr) public pure returns (uint160) {
        return uint160(addr);
    }
}
");



contract TypeConversions {
    // Conversion explicite
    function uintToInt(uint256 a) public pure returns (int256) {
        return int256(a);
    }

    // Conversion address to payable
    function toPayable(address addr) public pure returns (address payable) {
        return payable(addr);
    }

    // String to bytes
    function stringToBytes(string memory s) public pure returns (bytes memory) {
        return bytes(s);
    }

    // Address to uint (pour comparaison)
    function addressToUint(address addr) public pure returns (uint160) {
        return uint160(addr);
    }
}



### Interpretation : les conversions de types en Solidity

Les quatre conversions couvrent les cas les plus frequents :

- **`uint256 -> int256`** (explicite) conserve la valeur seulement de 0 à `2^255 - 1`.
  Au-delà, le même motif binaire est interprété comme négatif :
  `2^255` devient `-2^255`, et `2^256 - 1` devient `-1`.
  Cette conversion explicite ne déclenche pas le revert des débordements
  arithmétiques de Solidity 0.8 : vérifiez la borne avant de convertir.
  La réciproque exige aussi une borne : un `int256` négatif devient
  un grand entier `uint256`.
- **`address -> address payable`** — nécessaire pour appeler `.transfer()` ou `.send()`. Depuis Solidity 0.8, `payable(addr)` est la syntaxe explicite ; la conversion inverse est automatique.
- **`string -> bytes`** — `bytes` est la representation bas-niveau ; la conversion est gratuite car `string` est déjà encodee en UTF-8 sous le capot.
- **`address -> uint160`** — une adresse Ethereum n'est qu'un entier de 160 bits ; cette conversion sert à **comparer ou trier** des adresses (mapping dense, arithmétique).

Toutes ces fonctions sont marquées `pure` : elles ne lisent ni n'ecrivent l'état du contrat, elles transforment juste leurs arguments. Un appel `pure` est donc **gratuit hors-transaction** (view call local, pas de frais de gaz).

Le contrat affiché dans la sortie (`TypeConversions`) rassemble les quatre conversions du quotidien, toutes `pure` donc appelables gratuitement en lecture. Le piège de la section 4 concerne les deux sens de conversion signée : une valeur hors plage peut changer de signe sans revert. Les exemples de bornes ci-dessus décrivent le langage ; le contrat imprimé n'a pas ete appele ici.

***

## 5. Exemples guidés et Exercices

Deux exemples guidés **entièrement résolus** précèdent les exercices : lisez leurs sorties et leurs analyses avant de commencer, ils montrent le pattern complet (déclaration, déploiement, appels, lecture des valeurs) que vous réutiliserez.

### Exemple guide 1 : Contrat SimpleStorage

Un contrat qui stocke et récupère une valeur uint256. Solution complète compilée et déployée.

Pattern minimal d'un contrat de stockage : une variable d'état `uint256 private data`, un `set()` qui écrit (transaction, donc du gaz), un `get()` en `view` (lecture gratuite). C'est la brique de base de la plupart des contrats.

In [11]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SimpleStorage {
    uint256 private data;

    function set(uint256 _value) public {
        data = _value;
    }

    function get() public view returns (uint256) {
        return data;
    }
}
";

 var c = await Deploy(source,"SimpleStorage");

Console.WriteLine($"initial = {await c.GetFunction("get").CallAsync<BigInteger>()}");
 await Write(c,"set",new BigInteger(42));
 Console.WriteLine($"après 42 = {await c.GetFunction("get").CallAsync<BigInteger>()}");
 await Write(c,"set",new BigInteger(1337));
 Console.WriteLine($"après 1337 = {await c.GetFunction("get").CallAsync<BigInteger>()}");

}

SimpleStorage déployé : 0xa51c1fc2f0d1a1b8494ed1fe312d7c3a78ed91c0, tx 0x945f76cc10c870194b14228ff00a973ca3d0d37afcf38ae38d134beefd61aefc, gas 117683


initial = 0


set : tx 0x7da0f6606aa29a1d47c3dfc35084c1a36052b977a9d7efc5df3cb3e8002e71fc, bloc 14, gas 43696


après 42 = 42


set : tx 0x1058826dc5ff8fb5d6d9e852103fc78ceee8d87729e175b77ad2840ade8f9709, bloc 15, gas 26608


après 1337 = 1337


**Analyse** : Le contrat `SimpleStorage` utilise une variable d'état `uint256 private data` pour stocker une valeur. La fonction `set()` modifie le storage (coûte du gas), tandis que `get()` est `view` (lecture seule, gratuite). La valeur initiale d'un `uint256` est 0 (valeur par defaut).

### Exemple guide 2 : Contrat Owner

Un contrat qui stocke l'adresse du propriétaire et permet de la changer. Solution complète compilée et déployée.

Le pattern **ownership** : une variable d'état `owner`, initialisée dans le `constructor` avec `msg.sender`, et une fonction `changeOwner` qui la met à jour. C'est le socle du contrôle d'accès — dans sa forme sûre, `changeOwner` exige `require(msg.sender == owner)`.

In [12]:
{
 var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Owner {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    function changeOwner(address _newOwner) public {
        owner = _newOwner;
    }
}
";

 var c = await Deploy(source,"Owner");

Console.WriteLine($"owner initial = {await c.GetFunction("owner").CallAsync<string>()}; déployeur = {deployer}");
 await Write(c,"changeOwner",accounts[1]);
 Console.WriteLine($"nouvel owner = {await c.GetFunction("owner").CallAsync<string>()}; attendu = {accounts[1]}");

}

Owner déployé : 0x0b306bf915c4d645ff596e518faf3f9669b97016, tx 0x8dc7a0e2eebb33769866ebebbc371a1a1a4df04b8f497b1a2a3ba660e8052316, gas 173099


owner initial = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266; déployeur = 0xf39fd6e51aad88f6f4ce6ab8827279cfffb92266


changeOwner : tx 0x523cb9a12a9eb0067a37a2dfcc5507a1442fad5198e1f8af4a20f48dae60eae2, bloc 17, gas 26946


nouvel owner = 0x70997970C51812dc3A010C7d01b50e0d17dc79C8; attendu = 0x70997970c51812dc3a010c7d01b50e0d17dc79c8


**Analyse** : Le contrat `Owner` utilise le pattern ownership : le constructor capture `msg.sender` (l'adresse du deployeur) comme owner initial. La fonction `changeOwner()` met à jour cette adresse. Notez que dans cette version simplifiee, n'importe qui peut changer l'owner — un vrai contrat ajouterait un `require(msg.sender == owner)`.

### Exercice 1 : Contrat CounterEvents

Créez un compteur qui emet un événement a chaque incrementation. Les events sont le mécanisme de logging on-chain en Solidity.

**Objectifs** :
1. Définir un `event` avec des paramètres indexes
2. Emettre un event avec `emit` dans une fonction
3. Capturer l'event dans les logs de transaction

In [13]:
{
// Exercice 1 : Contrat CounterEvents
// TODO etudiant : implementer un compteur qui emet un evenement a chaque incrementation
// Etape 1 : Definir un event Counted(address indexed caller, uint256 newCount)
// Etape 2 : Implementer increment() qui emet l'event
// Etape 3 : Implementer getCount() view returns (uint256)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract CounterEvents {
    // TODO etudiant : event, variable d'etat, et fonctions
}
";

// var c = await Deploy(source, "CounterEvents");
// await Write(c, "increment");
// Console.WriteLine(await c.GetFunction("getCount").CallAsync<BigInteger>());
Console.WriteLine("Exercice à compléter : CounterEvents");

}

Exercice à compléter : CounterEvents



(7,5): warning CS0219: The variable 'source' is assigned but its value is never used



### Exercice 2 : Contrat OwnedCounter

Combinez les concepts des exemples précédents : créez un compteur qui ne peut être incremente que par le propriétaire du contrat.

**Objectifs** :
1. Combiner variables d'état (`uint256`, `address`) et contrôle d'accès
2. Utiliser `require()` pour proteger une fonction
3. Deployer et tester le contrôle d'ownership sur anvil

In [14]:
{
// Exercice 2 : Contrat OwnedCounter
// TODO etudiant : implementer un compteur avec controle d'ownership
// Etape 1 : Definir une variable d'etat uint256 public count et address public owner
// Etape 2 : Capturer msg.sender dans le constructor
// Etape 3 : Implementer increment() avec require(msg.sender == owner, "Not owner")
// Etape 4 : Implementer getCount() view returns (uint256)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract OwnedCounter {
    // TODO etudiant : variables d'etat et fonctions
}
";

// var c = await Deploy(source, "OwnedCounter");
// await Write(c, "increment");
// Console.WriteLine(await c.GetFunction("getCount").CallAsync<BigInteger>());
Console.WriteLine("Exercice à compléter : OwnedCounter");

}

Exercice à compléter : OwnedCounter



(8,5): warning CS0219: The variable 'source' is assigned but its value is never used



### Exercice 3 : Contrat NumberBox avec conversions de types

Mettez en pratique les conversions de types (section 4) en creant un contrat qui stocke
un entier et expose ses representations dans plusieurs types.

**Objectifs** :
1. Stocker `int256 public value`
2. Implementer `setValue(int256 _v)`
3. Implementer `toUint() public view returns (uint256)` avec conversion `uint256(value)`

**Indice** : `uint256(value)` convertit explicitement. Convertir un negatif donne un très grand nombre.

In [15]:
{
// Exercice 3 : Contrat NumberBox avec conversions de types
// TODO etudiant : implementer un contrat qui stocke un int256 et le convertit en uint256
// Etape 1 : Definir int256 public value
// Etape 2 : Implementer setValue(int256 _v)
// Etape 3 : Implementer toUint() public view returns (uint256) avec uint256(value)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract NumberBox {
    // TODO etudiant : variable d'etat int256, setValue et toUint
}
";

// var c = await Deploy(source, "NumberBox");
// await Write(c, "setValue", new BigInteger(42));
// Console.WriteLine(await c.GetFunction("toUint").CallAsync<BigInteger>());
Console.WriteLine("Exercice à compléter : NumberBox");

}

Exercice à compléter : NumberBox



(7,5): warning CS0219: The variable 'source' is assigned but its value is never used



### Exercice 4 : Contrat MessageStore (string et bytes)

Reutilisez les types `string` / `bytes` (section 2.4) pour créer un contrat qui stocke
un message et expose sa longueur en octets.

**Indice** : `bytes(message).length` donne la longueur en octets (pas en caractères).

In [16]:
{
// Exercice 4 : Contrat MessageStore (string et bytes)
// TODO etudiant : implementer un contrat qui stocke un string et expose sa longueur en octets
// Etape 1 : Definir string public message
// Etape 2 : Implementer setMessage(string memory _msg)
// Etape 3 : Implementer byteLength() view returns (uint256) avec bytes(message).length
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract MessageStore {
    // TODO etudiant : variable d'etat string, setMessage et byteLength
}
";

// var c = await Deploy(source, "MessageStore");
// await Write(c, "setMessage", "Solidity");
// Console.WriteLine(await c.GetFunction("byteLength").CallAsync<BigInteger>());
Console.WriteLine("Exercice à compléter : MessageStore");

}

Exercice à compléter : MessageStore



(7,5): warning CS0219: The variable 'source' is assigned but its value is never used



### Exercice 5 : Contrat Toggle (booléens et variables d'état)

Combinez booléens (section 2.1) et variables d'état (section 3.1) pour créer un
interrupteur on/off avec compteur de bascules.

**Indice** : `isOn = !isOn` inverse le booléen. `returns (bool, uint256)` pour multi-return.

In [17]:
{
// Exercice 5 : Contrat Toggle (booleens et variables d'etat)
// TODO etudiant : implementer un interrupteur on/off avec compteur de bascules
// Etape 1 : Definir bool public isOn et uint256 public toggleCount
// Etape 2 : Implementer toggle() qui inverse isOn et incremente toggleCount
// Etape 3 : Implementer status() view returns (bool, uint256)
var source = @"
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Toggle {
    // TODO etudiant : variables d'etat bool/uint256, toggle et status
}
";

// var c = await Deploy(source, "Toggle");
// await Write(c, "toggle");
// await Write(c, "toggle");
// var status = await c.GetFunction("status").CallDecodingToDefaultAsync();
// Console.WriteLine(string.Join(", ", status.Select(x => x.Result)));
Console.WriteLine("Exercice à compléter : Toggle");

}

Exercice à compléter : Toggle



(7,5): warning CS0219: The variable 'source' is assigned but its value is never used



## Vérifiez votre compréhension

1. Pourquoi l’adresse du contrat n’est-elle utilisable qu’après le receipt de déploiement ?
2. Pourquoi `calculate` lu avant une transaction ne suffit-il pas à modifier `result` ?
3. Que montrent les lectures successives de `SimpleStorage` ?
4. Pourquoi convertir un entier signé négatif vers `uint256` exige-t-il une vérification préalable ?

<details><summary>Éléments de réponse</summary>

1. Le receipt miné confirme le succès et porte l’adresse de l’instance créée.
2. `eth_call` exécute sur une copie de l’état ; la transaction minée est nécessaire pour persister l’affectation.
3. Les lectures après les deux écritures montrent les valeurs enregistrées, d’abord 42 puis 1337.
4. La conversion réinterprète les bits et peut produire un grand entier non signé sans provoquer le revert arithmétique.

</details>

## Pour aller plus loin

Reprenez le contrat `Owner` et étudiez la différence entre une adresse conservée comme donnée et une condition d’autorisation. Le contrat guidé permet de changer le propriétaire sans restriction ; ajouter un contrôle d’accès est précisément l’objet d’un exercice. La suite du parcours introduit les modifiers pour exprimer ce contrôle et les fonctions payables pour transférer des fonds.

***

## 6. Résumé

| Type | Description | Exemple |
|------|-------------|---------|
| `bool` | Booléen | `true`, `false` |
| `uint256` | Entier non signe | `42`, `1000000` |
| `int256` | Entier signe | `-42`, `42` |
| `address` | Adresse Ethereum | `0x1234...` |
| `bytes32` | Bytes fixes | `hex"abcd"` |
| `string` | Chaîne UTF-8 | `"Hello"` |

***

**Notebook suivant** : [SC-04-Functions-State-Python](SC-04-Functions-State-Python.ipynb)

Ce tableau récapitule les value types manipulés dans ce notebook ; retenez surtout la règle des conversions (section 4) et la dualité code immuable / état mutable (section 1).

## Résumé et perspectives

Ce notebook a posé les fondations du langage Solidity en explorant ses types de données fondamentaux (booléens, entiers signés et non signés, adresses Ethereum, bytes et strings) et les mécanismes de variables (d'état persistantes en storage, locales temporaires en mémoire, globales comme `msg.sender` et `block.timestamp`). Les démonstrations exécutées sur la blockchain locale anvil illustrent le comportement des types et des opérateurs. Le contrat de conversion, lui, est présenté par son source sans exécution.

La distinction entre variables d'état et variables locales est centrale dans la programmation Solidity : les premières persistent sur la blockchain entre les appels de fonction mais coûtent du gas, tandis que les secondes sont gratuites mais temporaires. Les variables globales comme `msg.sender`, `msg.value` et `block.timestamp` relient le contexte d'exécution à la logique du contrat. `msg.sender` et `block.timestamp` sont lus dans la démonstration ; `getMsgValue` est défini mais reste à appeler pour observer `msg.value`. Les conversions de types (`uint256` vers `int256`, `address` vers `address payable`, `string` vers `bytes`) complètent ce socle en permettant les manipulations nécessaires à la logique métier.

Ce socle de types et de variables est le prerequisite direct pour le notebook suivant, [SC-04-Functions-State-Python](SC-04-Functions-State-Python.ipynb), qui approfondit les data locations (`storage`, `memory`, `calldata`), la visibilité des fonctions et les modificateurs (`view`, `pure`, `payable`), ainsi que les custom modifiers.

Ordre d'apprentissage conseillé, celui du notebook : les types d'abord (ils déterminent ce qui est stockable et à quel coût), puis les variables (storage vs mémoire — la décision qui pèse le plus sur le gaz), puis les conversions (le moment où les types se rencontrent). Les exercices 1 à 5 réutilisent exactement ces trois blocs.

## À retenir

Les types de valeur, les variables d’état et les lectures ABI permettent de raisonner sur les données Solidity. Une transaction doit être minée avant de vérifier l’état.

[Version Python](SC-03-Solidity-Basics-Python.ipynb) | [Fonctions et état](SC-04-Functions-State-CSharp.ipynb)